
# PhishU — Master Notebook (EDA • PCA • Tabular Models • Semantic Baselines)

This notebook orchestrates all analyses and models without changing individual scripts.

**Sections**  
1. Environment & Logging  
2. Data loading (single source of truth)  
3. EDA (src/data_analysis/EDA.py)  
4. PCA / ACP (src/data_analysis/ACP.py)  
5. Tabular models: Logistic Regression, Random Forest, XGBoost (src/models/*.py)  
6. Semantic baselines: TF-IDF+LinearSVC, FastText+LogReg (src/semantic_models/...)  
7. Summary comparison table


## 1) Environment & Logging

This section prepares the runtime for reproducible experiments and clear tracing:
- Initializes the logging system (init_logging). Change the level to "DEBUG" for more verbose output.
- Retrieves a notebook-scoped logger (get_logger) used across cells to record progress and diagnostics.
- Sets a global random seed (set_seed) so data splits, sampling and model training are repeatable when possible.

Use a more verbose logging level while debugging; keep INFO for regular runs.

In [3]:
from src.pipeline.logger import init_logging, get_logger, set_seed
init_logging("INFO")  # set "DEBUG" for more verbose logs
log = get_logger("Notebook", "INFO")
set_seed(42)
log.info("Notebook started.")


[21:33:58] [INFO] Notebook: Notebook started.


## 2) Data loading

This cell loads the PhiUSIIL Phishing URL dataset from UCI via the project DataLoader and prepares a single DataFrame for downstream analysis.

- Data source: DataLoader(dataset_id=967) → returns X_all (features), y_series (labels) and meta (dataset metadata including UCI id, name and data URL).
- Construction: df = X_all.copy(); df["label"] = y_series.astype(int) (adds numeric label column).
- Dataset size & schema (from the cell output): 235,795 rows and 55 columns (54 original features + 1 label). Dtypes: 41 int64, 10 float64, 4 object. Memory ~99 MB.
- Label balance: ~57.2% label=1 (phishing) vs ~42.8% label=0 (legitimate) (see eda_out["label_distribution_pct"]).
- Splits used later: the notebook uses an 80/20 split for experiments (Xtr ≈ 188,636, Xte ≈ 47,159).
- meta: contains dataset metadata and provenance (useful for reproducibility and citation).

The DataFrame df is now ready for EDA, PCA and the tabular/semantic models that follow.

In [4]:

# Load from UCI via our shared DataLoader
from src.utils.data_loader import DataLoader

dl = DataLoader(dataset_id=967)  # PhiUSIIL Phishing URL dataset
X_all, y_series, meta = dl.get_xy_as_dataframes()

# Build a single DataFrame
df = X_all.copy()
df["label"] = y_series.astype(int).values  # ensure numeric labels

log.info(f"Loaded from UCI: {df.shape[0]} rows, {df.shape[1]} cols "
         f"(features={X_all.shape[1]})")
df.head(3)

[21:34:06] [INFO] Notebook: Loaded from UCI: 235795 rows, 55 cols (features=54)


,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,...,0,0,1,34,20,28,119,0,124,1
1,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,...,0,0,1,50,9,8,39,0,217,1
2,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,...,0,0,1,10,2,7,42,2,5,1


## 3) EDA

Purpose: run a quick sanity check to verify schema, missingness and class balance before feature engineering and modeling.

Outputs produced:
- Shape & dtypes summary (235,795 × 55; mostly numeric).
- Missing-values check (none).
- Label balance (~57.2% phishing / 42.8% legitimate).
- Numeric preview (summary statistics) and a saved label-distribution plot: outputs/eda/distribution_label.png.


In [5]:

import pandas as pd
from src.data_analysis.EDA import run_eda

eda_out = run_eda(
    df=df,
    label_col="label",
    save_dir="outputs/eda",
    save_fig=True,
    show_fig=False
)

display(pd.Series(eda_out["shape"], index=["rows","cols"]).to_frame("shape"))
display(eda_out["dtypes_counts"].to_frame("count").T)
display(eda_out["label_distribution_pct"].to_frame("pct"))
if eda_out["preview_describe"].shape[0] > 0:
    display(eda_out["preview_describe"])


[21:34:08] [INFO] EDA: ▶ Describing preview columns (5 cols) ...


[21:34:08] [INFO] EDA: ✓ Describing preview columns (5 cols) done in 0.1s
[21:34:08] [INFO] EDA: ▶ Creating label distribution plot ...
[21:34:08] [INFO] EDA: Figure saved to: outputs/eda\distribution_label.png
[21:34:08] [INFO] EDA: ✓ Creating label distribution plot done in 0.2s


,shape
rows,235795
cols,55


,int64,float64,object
count,41,10,4


,pct
label,
1,57.189508
0,42.810492


,URLLength,DomainLength,NoOfJS,NoOfImage,NoOfExternalRef
count,235795.000000,235795.000000,235795.000000,235795.000000,235795.000000
mean,34.573095,21.470396,10.522305,26.075689,49.262516
std,41.314153,9.150793,22.312192,79.411815,161.027430
min,13.000000,4.000000,0.000000,0.000000,0.000000
25%,23.000000,16.000000,0.000000,0.000000,1.000000
50%,27.000000,20.000000,6.000000,8.000000,10.000000
75%,34.000000,24.000000,15.000000,29.000000,57.000000
max,6097.000000,110.000000,6957.000000,8956.000000,27516.000000


## 4) PCA
We run PCA on the numeric feature set (StandardScaler → PCA(n_components=0.95)). Key results and how to read them:

- Retained 36 principal components to explain ≈95% of the variance (X_pca shape: 235,795 × 36).
- StandardScaler was applied before PCA; the PCA object and scaler are available in pca_out for reproducibility.
- Components table: components_df contains the loadings for each PC (36 rows × ~50 original numeric features). Inspecting loadings helps interpret what each PC captures.
- PC1 is dominated by lexical/domain similarity and social signals (top loadings include URLSimilarityIndex, HasSocialNet, DomainTitleMatchScore), suggesting a mix of URL–title/domain alignment and presence of social links.
- PC2 is driven by syntactic/token counts and special characters (top loadings include NoOfEqualsInURL, NoOfDegitsInURL, URLLength), indicating a dimension related to punctuation/digit-heavy or obfuscated URLs.
- Plots saved to outputs/pca:
    - Scree plot: outputs/pca/pca_scree.png — use to verify explained-variance elbow.
    - PC1 vs PC2 scatter: outputs/pca/pca_scatter_pc1_pc2.png — useful to visualize class separation or outliers.

In [6]:

from src.data_analysis.ACP import run_pca

pca_out = run_pca(
    df=df,
    label_col="label",
    n_components=0.95,
    save_dir="outputs/pca",
    save_fig=True,
    show_fig=False
)

log.info(f"PCA retained components: {pca_out['n_components_']} "
         f"(cumulative variance={pca_out['explained_variance_ratio'].sum():.3f})")
display(pca_out["components_df"].head(5))
display(pd.DataFrame({
    "PC1_top": pca_out["pc1_top_loadings"],
    "PC2_top": pca_out["pc2_top_loadings"]
}))


[21:34:14] [INFO] ACP: ▶ Selecting numeric columns ...
[21:34:14] [INFO] ACP: ✓ Selecting numeric columns done in 0.1s
[21:34:14] [INFO] ACP: ▶ Scaling features (StandardScaler) ...
[21:34:15] [INFO] ACP: ✓ Scaling features (StandardScaler) done in 0.2s
[21:34:15] [INFO] ACP: ▶ Fitting PCA (n_components=0.95) ...
[21:34:15] [INFO] ACP: ✓ Fitting PCA (n_components=0.95) done in 0.1s
[21:34:15] [INFO] ACP: Retained components: 36
[21:34:15] [INFO] ACP: Cumulative explained variance: 0.9507
[21:34:15] [INFO] ACP: ▶ Building components (loadings) DataFrame ...
[21:34:15] [INFO] ACP: ✓ Building components (loadings) DataFrame done in 0.0s
[21:34:15] [INFO] ACP: ▶ Creating Scree plot (cumulative explained variance) ...
[21:34:15] [INFO] ACP: ✓ Creating Scree plot (cumulative explained variance) done in 0.2s
[21:34:15] [INFO] ACP: ▶ Creating 2D scatter on first two PCs ...
[21:34:25] [INFO] ACP: ✓ Creating 2D scatter on first two PCs done in 9.8s
[21:34:25] [INFO] Notebook: PCA retained compo

,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
PC1,-0.170095,-0.134718,-0.058771,0.286119,0.207192,0.072324,0.182087,0.000095,-0.068942,-0.044377,...,0.084618,0.134157,0.044735,0.224363,0.111093,0.029155,0.133114,0.125423,0.045029,0.104293
PC2,0.324775,0.020200,0.125017,0.017056,0.008703,0.026520,-0.005669,-0.000445,0.009367,0.130631,...,0.081272,0.106953,0.031041,0.136251,0.078345,0.022841,0.085552,0.086069,0.035035,0.072569
PC3,0.028841,-0.231840,-0.025855,0.085568,0.318810,0.193648,0.117737,0.162269,-0.309943,0.091201,...,-0.139955,-0.133901,-0.037968,-0.066753,-0.140806,-0.049191,-0.097251,-0.168148,-0.069285,-0.142677
PC4,0.160260,0.256281,0.035134,-0.130614,0.004626,0.366830,0.109629,0.363739,-0.173579,-0.253843,...,0.087275,0.087231,0.026008,-0.022263,0.005337,-0.002240,-0.003534,-0.048682,0.008108,-0.041230
PC5,-0.056261,-0.005228,-0.116075,-0.075127,0.062607,0.107745,-0.014844,0.097087,-0.092080,0.203110,...,0.188448,0.117396,0.083099,-0.119841,0.279052,0.106518,0.003368,0.298222,0.112660,0.268772


,PC1_top,PC2_top
CharContinuationRate,0.207192,NaN
DomainTitleMatchScore,0.230340,NaN
HasCopyrightInfo,0.224363,NaN
HasDescription,0.216431,NaN
HasSocialNet,0.241459,NaN
HasSubmitButton,0.188332,0.147001
IsHTTPS,NaN,0.152316
NoOfAmpersandInURL,NaN,0.308973
NoOfDegitsInURL,NaN,0.343919
NoOfEqualsInURL,NaN,0.350142


## 5) Tabular models
A compact summary of the tabular experiments and key observations.

- Setup
    - All tabular models use the same compact lexical feature set (12 features such as URLLength, NoOfSubDomain, NoOfOtherSpecialCharsInURL, TLDLength, …).
    - LogisticRegression / RandomForest / XGBoost were run on a 10k sampled experiment (n_train≈8k / n_test≈2k). The Neural Network was trained on the full 80/20 split (n_train=188,636 / n_test=47,159).
    - Per-run artifacts (confusion matrices, coefficient/importance plots, training curves) are saved under outputs/<model_name>.

- Logistic Regression (baseline)
    - Interpretable linear baseline with coefficients and top positive/negative predictors saved.
    - Good recall (≈0.98) but lower precision (≈0.81) — model favors catching phishing at the cost of more false positives.
    - Useful for understanding which lexical signals push the decision.

- Random Forest
    - Non‑linear ensemble improves over LR (accuracy ≈0.8785, roc_auc ≈0.973).
    - Feature importances highlight token/special-character signals (NoOfOtherSpecialCharsInURL among top contributors).
    - Offers better precision/overall balance versus LR on the sampled run.

- XGBoost
    - Strong boost in predictive performance on the sampled run (accuracy ≈0.983, roc_auc ≈0.996).
    - Importance profiles show the model relies heavily on a few lexical features — inspect importances to detect dominance/instability.
    - High scores warrant checking for overfitting and calibration (PR‑AUC for sampled tabular runs is not computed here).

- Neural Network
    - MLP on scaled features trained on the full split achieves the best numbers (accuracy ≈0.9855, roc_auc ≈0.9969).
    - Training report and confusion matrix are saved; gives strong class separation on the provided split.
    - As with XGBoost, validate generalization, decision calibration and sensitivity to sampling.

- Takeaway
    - Tree boosting and the NN outperform the linear baseline on these lexical/tabular features and also beat semantic baselines in raw metrics (see summary table).

In [9]:

from src.models.regressionlogistique import run_logistic_regression
from src.models.randomforest import run_random_forest
from src.models.XGboost import run_xgboost
from src.models.NN import run_neural_network

tabular_results = {}
common_features = (
    'URLLength','DomainLength','NoOfSubDomain','IsDomainIP',
    'NoOfLettersInURL','NoOfDegitsInURL','NoOfEqualsInURL',
    'NoOfQMarkInURL','NoOfAmpersandInURL','NoOfOtherSpecialCharsInURL',
    'SpacialCharRatioInURL','TLDLength'
)

log.info("Running Logistic Regression (tabular)...")
logreg_out = run_logistic_regression(
    df=df,
    label_col="label",
    features=common_features,
    sample_n=10000,
    save_dir="outputs/logreg",
    save_fig=True,
    show_fig=False
)
tabular_results["LogisticRegression"] = logreg_out["metrics"]
display(pd.Series(logreg_out["metrics"], name="LogisticRegression"))

log.info("Running Random Forest (tabular)...")
rf_out = run_random_forest(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    sample_n=10000,
    save_dir="outputs/random_forest",
    save_fig=True,
    show_fig=False
)
tabular_results["RandomForest"] = rf_out["metrics"]
display(pd.Series(rf_out["metrics"], name="RandomForest"))

log.info("Running XGBoost (tabular)...")
xgb_out = run_xgboost(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    sample_n=10000,
    save_dir="outputs/xgboost",
    save_fig=True,
    show_fig=False
)
tabular_results["XGBoost"] = xgb_out["metrics"]
display(pd.Series(xgb_out["metrics"], name="XGBoost"))

log.info("Running Neural Network (tabular)...")
nn_out = run_neural_network(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    early_stop=True,
    save_dir="outputs/neural_network",
    save_fig=True,
    show_fig=False
)
tabular_results["NeuralNetwork"] = nn_out["metrics"]
display(pd.Series(nn_out["metrics"], name="NeuralNetwork"))


[21:39:11] [INFO] Notebook: Running Logistic Regression (tabular)...
[21:39:11] [INFO] LogRegTab: ▶ Stratified sampling to n=10000 ...
[21:39:12] [INFO] LogRegTab: ✓ Stratified sampling to n=10000 done in 0.2s
[21:39:12] [INFO] LogRegTab: Échantillon sélectionné : 10000 lignes
[21:39:12] [INFO] LogRegTab: Variables conservées (12 au total) : ['URLLength', 'DomainLength', 'NoOfSubDomain', 'IsDomainIP', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'TLDLength']
[21:39:12] [INFO] LogRegTab: ▶ Scaling features (StandardScaler) ...
[21:39:12] [INFO] LogRegTab: ✓ Scaling features (StandardScaler) done in 0.0s
[21:39:12] [INFO] LogRegTab: ▶ Training LogisticRegression ...
[21:39:12] [INFO] LogRegTab: ✓ Training LogisticRegression done in 0.0s
[21:39:12] [INFO] LogRegTab: ▶ Scoring ...
[21:39:12] [INFO] LogRegTab: ✓ Scoring done in 0.0s
[21:39:12] [INFO] LogRegTab: Modèle basé sur 10 000 

accuracy     0.860500
precision    0.813633
recall       0.980769
f1           0.889417
roc_auc      0.938074
Name: LogisticRegression, dtype: float64

[21:39:13] [INFO] Notebook: Running Random Forest (tabular)...
[21:39:13] [INFO] RandomForestTab: ▶ Stratified sampling to n=10000 ...
[21:39:13] [INFO] RandomForestTab: ✓ Stratified sampling to n=10000 done in 0.1s
[21:39:13] [INFO] RandomForestTab: Using 12 features: ['URLLength', 'DomainLength', 'NoOfSubDomain', 'IsDomainIP', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'TLDLength']
[21:39:13] [INFO] RandomForestTab: ▶ Scaling features (StandardScaler) ...
[21:39:13] [INFO] RandomForestTab: ✓ Scaling features (StandardScaler) done in 0.0s
[21:39:13] [INFO] RandomForestTab: ▶ Training RandomForest ...
[21:39:13] [INFO] RandomForestTab: ✓ Training RandomForest done in 0.3s
[21:39:13] [INFO] RandomForestTab: ▶ Scoring ...
[21:39:13] [INFO] RandomForestTab: ✓ Scoring done in 0.1s
[21:39:13] [INFO] RandomForestTab: Acc=0.8785 | Prec=0.8390 | Rec=0.9747 | F1=0.9017 | AUC=0.9734
[21:

accuracy     0.878500
precision    0.838977
recall       0.974650
f1           0.901739
roc_auc      0.973436
Name: RandomForest, dtype: float64

[21:39:14] [INFO] Notebook: Running XGBoost (tabular)...
[21:39:14] [INFO] XGBoostTab: ▶ Stratified sampling to n=10000 ...
[21:39:14] [INFO] XGBoostTab: ✓ Stratified sampling to n=10000 done in 0.2s
[21:39:14] [INFO] XGBoostTab: Using 12 features: ['URLLength', 'DomainLength', 'NoOfSubDomain', 'IsDomainIP', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'TLDLength']
[21:39:14] [INFO] XGBoostTab: ▶ Scaling features (StandardScaler) ...
[21:39:14] [INFO] XGBoostTab: ✓ Scaling features (StandardScaler) done in 0.0s
[21:39:14] [INFO] XGBoostTab: ▶ Training XGBClassifier ...
[21:39:14] [INFO] XGBoostTab: ✓ Training XGBClassifier done in 0.4s
[21:39:14] [INFO] XGBoostTab: ▶ Scoring ...
[21:39:14] [INFO] XGBoostTab: ✓ Scoring done in 0.0s
[21:39:14] [INFO] XGBoostTab: Acc=0.9830 | Prec=0.9826 | Rec=0.9878 | F1=0.9852 | AUC=0.9962
[21:39:14] [INFO] XGBoostTab: ▶ Rendering confusion matrix

...

 Total params: 12,033 (47.00 KB)

 Trainable params: 12,033 (47.00 KB)

 Non-trainable params: 0 (0.00 B)

[21:39:15] [INFO] NeuralNetworkTab: ✓ Building Neural Network model done in 0.1s
[21:39:15] [INFO] NeuralNetworkTab: ▶ Compiling Neural Network model ...
[21:39:15] [INFO] NeuralNetworkTab: ✓ Compiling Neural Network model done in 0.0s
[21:39:15] [INFO] NeuralNetworkTab: ▶ Training Neural Network model ...
Epoch 1/100
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.8393 - loss: 0.3790 - val_accuracy: 0.8554 - val_loss: 0.3508
Epoch 2/100
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8590 - loss: 0.3458 - val_accuracy: 0.8680 - val_loss: 0.3357
Epoch 3/100
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8625 - loss: 0.3389 - val_accuracy: 0.8662 - val_loss: 0.3282
Epoch 4/100
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8676 - loss: 0.3232 - val_accuracy: 0.8492 - val_loss: 0.3539
Epoch 5/100
2948/2948 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.8861 - loss: 0.2770 - val_accuracy: 0.9402 - val_loss: 0.2147
Epoch 6/100
2948/2948 ━━━━━━━━━━━━━━━

accuracy     0.985517
precision    0.978799
recall       0.996255
f1           0.987450
roc_auc      0.996883
Name: NeuralNetwork, dtype: float64

## 6) Semantic baselines (TF-IDF+LinearSVC, FastText+LogReg)
- What we ran  
    - Two semantic baselines on the same 80/20 split: (1) char-level TF‑IDF (char n‑grams 3–5, max 50k) + LinearSVC, and (2) URL tokenization → FastText embeddings + LogisticRegression. The notebook saves artifacts (vectorizer, classifier, embedder, token transformer, confusion matrices and top n-grams/tokens) in sem_out.

- Short summary of key outputs  
    - Char TF‑IDF + LinearSVC: accuracy ≈ 0.903, precision ≈ 0.8785, recall ≈ 0.9637, f1 ≈ 0.9192, roc_auc ≈ 0.9424, pr_auc ≈ 0.9390. Top discriminative character n‑grams were printed from the TF‑IDF model.  
    - FastText + LogReg: accuracy ≈ 0.8734, precision ≈ 0.8561, recall ≈ 0.9360, f1 ≈ 0.8943, roc_auc ≈ 0.9237, pr_auc ≈ 0.9199. FastText vocab ≈ 8.6k; the notebook prints class‑distinctive tokens and nearest neighbors.

- Interpretation / takeaway (short)  
    - Both semantic baselines achieve high recall (they favour catching phishing URLs) and are reasonably strong (TF‑IDF beats FastText on all reported metrics). TF‑IDF captures discriminative character substrings (good for obfuscated/lexical cues), while FastText provides token-level semantic clusters useful for robustness and nearest‑neighbor inspection.  
    - Compared to the best tabular models (XGBoost / NN with accuracy ≈ 0.98+), semantic models are weaker in raw metrics here, but they offer complementary, interpretable signals (ngrams/tokens and embeddings) that can help diagnostics or be combined with tabular features.

In [10]:
from src.semantic_models.phishing_url_semantic_baselines import (
    train_and_compare_semantic_baselines, PhiUSIILSpec
)

# Pass the same df already loaded earlier in the notebook:
sem_out = train_and_compare_semantic_baselines(
    spec=PhiUSIILSpec(dataset_id=967),
    df=df,
    test_size=0.2,
    random_state=42,
    run_char_tfidf=True,
    run_fasttext=True,
    fasttext_fast_mode=True
)

# Build a compact metrics table
def _extract_metrics(res_dict):
    keep = ("accuracy","precision","recall","f1","roc_auc","pr_auc")
    table = {}
    for k, v in res_dict.items():
        if k in ("char_tfidf_svc", "fasttext_logreg"):
            table[k] = {m: float(v.get(m, float("nan"))) for m in keep}
    return table

sem_metrics = _extract_metrics(sem_out)
import pandas as pd
pd.DataFrame(sem_metrics).T


[21:40:57] [INFO] Semantic: [Data] Loading URLs & labels
[21:40:57] [INFO] Semantic: [Data] Train/test split
[21:40:57] [INFO] Semantic: [Data] Train=188636 | Test=47159
[21:40:57] [INFO] Semantic: [TF-IDF+SVC] Cleaning & vectorizing


Cleaning URLs (char-ngrams):   0%|          | 0/188636 [00:00<?, ?it/s]

Cleaning URLs (char-ngrams):   0%|          | 0/47159 [00:00<?, ?it/s]

[21:41:14] [INFO] Semantic: [TF-IDF+SVC] Training LinearSVC
[21:41:19] [INFO] Semantic: [TF-IDF+SVC] Evaluating

=== TF-IDF char (3–5) + LinearSVC ===
ROC-AUC: 0.9424
PR-AUC : 0.9390
Acc: 0.9031 | Prec: 0.8785 | Rec: 0.9637 | F1: 0.9192

-- Classification report --
              precision    recall  f1-score   support

           0      0.944     0.822     0.879     20189
           1      0.879     0.964     0.919     26970

    accuracy                          0.903     47159
   macro avg      0.911     0.893     0.899     47159
weighted avg      0.907     0.903     0.902     47159

-- Confusion matrix --
[[16595  3594]
 [  978 25992]]
[21:41:19] [INFO] Semantic: [FastText+LR] Tokenizing train/test URLs


Tokenizing URLs (word tokens):   0%|          | 0/188636 [00:00<?, ?it/s]

...

[21:42:18] [INFO] Semantic: [FastText+LR] Embedding FULL train/test sets


Averaging embeddings:   0%|          | 0/188636 [00:00<?, ?it/s]

Averaging embeddings:   0%|          | 0/47159 [00:00<?, ?it/s]

[21:42:33] [INFO] Semantic: [FastText+LR] Training LogisticRegression
[21:42:34] [INFO] Semantic: [FastText+LR] Evaluating

=== FastText (moyenne) + LogisticRegression ===
ROC-AUC: 0.9237
PR-AUC : 0.9199
Acc: 0.8734 | Prec: 0.8561 | Rec: 0.9360 | F1: 0.8943

-- Classification report --
              precision    recall  f1-score   support

           0      0.902     0.790     0.842     20189
           1      0.856     0.936     0.894     26970

    accuracy                          0.873     47159
   macro avg      0.879     0.863     0.868     47159
weighted avg      0.876     0.873     0.872     47159

-- Confusion matrix --
[[15946  4243]
 [ 1727 25243]]


,accuracy,precision,recall,f1,roc_auc,pr_auc
char_tfidf_svc,0.903051,0.878524,0.963737,0.919160,0.942446,0.939029
fasttext_logreg,0.873407,0.856101,0.935966,0.894254,0.923728,0.919917


In [12]:
import numpy as np
tfidf = sem_out["char_tfidf_svc"]["vectorizer"]
svc   = sem_out["char_tfidf_svc"]["clf"]
names = np.array(tfidf.get_feature_names_out())
coefs = svc.coef_[0]

idx_pos = np.argsort(coefs)[-10:][::-1]
idx_neg = np.argsort(coefs)[:10]

print("Top phishing n-grams:", ", ".join(names[idx_pos]))
print("Top legitimate n-grams:", ", ".join(names[idx_neg]))

Top phishing n-grams: .gal, .mil, .gov, .ac., .gr, imate, .com, .it, .hr, .jp
Top legitimate n-grams: .cf, om/, com/, .com/, .ml, .ga, .gq, .top, .fr/, fr/


In [13]:
from collections import Counter
from src.semantic_models.phishing_url_semantic_baselines import (
    URLWordTokenTransformer, load_urls_and_labels, PhiUSIILSpec
)
from sklearn.model_selection import train_test_split

# Recover same split and tokenization
urls_all, y_all, _ = load_urls_and_labels(PhiUSIILSpec(dataset_id=967), df=df)
Xtr, Xte, ytr, yte = train_test_split(urls_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

tok_tf = sem_out["fasttext_logreg"]["token_transformer"]
ft_model = sem_out["fasttext_logreg"]["embedder"].model

tokens_test = tok_tf.transform(Xte)
y_test_np = yte.to_numpy()

# --- Token frequencies per class ---
phish_counter, legit_counter = Counter(), Counter()
for toks, lab in zip(tokens_test, y_test_np):
    toks = [t for t in toks if t in ft_model.wv and t.isalnum() and len(t) > 2]
    if lab == 1:
        phish_counter.update(toks)
    else:
        legit_counter.update(toks)

# Remove tokens that appear in both (uninformative overlap)
shared_tokens = set(phish_counter.keys()) & set(legit_counter.keys())
for token in shared_tokens:
    del phish_counter[token]
    del legit_counter[token]

# Now get top distinctive tokens
top_phish = [w for w, _ in phish_counter.most_common(20)]
top_legit = [w for w, _ in legit_counter.most_common(20)]

print(f"FastText vocab size: {len(ft_model.wv)}")
print("\nDistinct phishing-tilted tokens:", ", ".join(top_phish))
print("Distinct legitimate-tilted tokens:", ", ".join(top_legit))

# --- Nearest neighbors for a few representative tokens ---
def nn(token, k=5):
    try:
        return ", ".join([f"{w}({s:.2f})" for w, s in ft_model.wv.most_similar(token, topn=k)])
    except KeyError:
        return "(OOV)"

probe = top_phish[:3] + top_legit[:3]
print("\nNearest neighbors:")
for t in probe:
    print(f"  {t:>15s} -> {nn(t)}")


Tokenizing URLs (word tokens):   0%|          | 0/47159 [00:00<?, ?it/s]

FastText vocab size: 8574

Distinct phishing-tilted tokens: county, jewelry, schools, kids, jewelers, chicago, awards, village, tree, aviation, wine, council, aero, airport, university, utah, tips, york, independent, saint
Distinct legitimate-tilted tokens: ipfs, repl, pfs, att, godaddy, akc, https, blogspot, dweb, glitch, xsph, fleek, assets, webmail, signin, webwave, account, facebook, wixsite, weebly

Nearest neighbors:
           county -> council(0.91), count(0.85), courts(0.85), counter(0.81), countryside(0.81)
          jewelry -> jewels(0.95), jewelers(0.89), jewellery(0.89), jewellers(0.89), jeff(0.86)
          schools -> school(0.96), scholars(0.90), preschool(0.90), george(0.87), civic(0.86)
             ipfs -> pfs(0.86), mfs(0.83), afk(0.82), infura(0.79), ffs(0.78)
             repl -> reply(0.85), pl(0.80), rep(0.78), re(0.77), asd156(0.76)
              pfs -> flare(0.87), ipfs(0.86), mfs(0.85), infura(0.85), fur(0.85)


## 7) Summary comparison table
The table below aggregates key metrics (accuracy, precision, recall, F1, ROC‑AUC, PR‑AUC) for the tabular and semantic baselines.

- Key points
    - Best raw performance: XGBoost and NeuralNetwork (accuracy ≈ 0.983–0.986, ROC‑AUC ≈ 0.996).  
    - Logistic Regression: very high recall (~0.98) but lower precision (~0.81) — useful as an interpretable high‑recall baseline.  
    - Random Forest: better precision/F1 than LR, but below XGBoost/NN.  
    - Semantic baselines (char TF‑IDF, FastText+LogReg): high recall and useful interpretable/semantic signals, but weaker than the top tabular models.


In [14]:
import numpy as np
import pandas as pd

metrics = ["accuracy","precision","recall","f1","roc_auc","pr_auc"]

# --- Tabular models (PR-AUC may be missing → stays NaN) ---
tab_df = pd.DataFrame(tabular_results).T.reindex(columns=metrics)

# --- Semantic models (pull same metric set) ---
sem_df = pd.DataFrame({
    k: {m: sem_out[k].get(m, np.nan) for m in metrics}
    for k in sem_out
    if k in ("char_tfidf_svc","fasttext_logreg")
}).T

# --- Combine (all numeric; NaN where not available) ---
combined_df = pd.concat([tab_df, sem_df], axis=0)

# --- Nice formatting: 4 decimals for numbers, "—" for NaN ---
display(
    combined_df
      .style
      .format({col: "{:.4f}" for col in combined_df.columns}, na_rep="—")
      .set_caption("Overall model comparison (tabular + semantic)")
)


,accuracy,precision,recall,f1,roc_auc,pr_auc
LogisticRegression,0.8605,0.8136,0.9808,0.8894,0.9381,—
RandomForest,0.8785,0.8390,0.9747,0.9017,0.9734,—
XGBoost,0.9830,0.9826,0.9878,0.9852,0.9962,—
NeuralNetwork,0.9855,0.9788,0.9963,0.9874,0.9969,—
char_tfidf_svc,0.9031,0.8785,0.9637,0.9192,0.9424,0.9390
fasttext_logreg,0.8734,0.8561,0.9360,0.8943,0.9237,0.9199
